In [1]:
%pip install pandas transformers datasets torch scikit-learn seqeval gcsfs google-cloud-storage mlflow optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 106.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.9 MB/s eta 0

In [2]:
import os
import torch

# pipeline.yaml structure 
config = {
    "dataset": {
        "text_column": "review",
        "target_column": "adr_label"
    },
    "training": {
        "model_name": "emilyalsentzer/Bio_ClinicalBERT",
        "epochs": 3,
        "batch_size": 16
    },
    "paths": {
        "models": "./biobert_adr_model"
    },
    "mlflow": {
        "experiment_name": "adr-nlp"
    }
}




In [3]:
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    pipeline
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import mlflow

In [ ]:
from google.colab import auth
auth.authenticate_user()

# Use gs:// protocol for GCS buckets
DATA_PATH = "gs://adr-nlp/processed"

import pandas as pd
train_df = pd.read_csv(f"{DATA_PATH}/train.csv")

train_df.shape
train_df.columns

Index(['uniqueID', 'drugName', 'condition', 'review', 'rating', 'date',
       'usefulCount'],
      dtype='object')

In [ ]:
train_df.head()

,uniqueID,drugName,condition,review,rating,date,usefulCount
0,40496,Savella,ibromyalgia,i039m on day 6 and i already feel the differen...,8,11-Jul-09,83
1,208687,Belsomra,Insomnia,didn039t work for me,1,28-Oct-17,2
2,228186,Etonogestrel,Birth Control,in my xprinc th only good thing about implanon...,2,25-Oct-09,3
3,17163,Flexeril,Muscle Spasm,flxril shorttrm mmory loss and xtrmly vivid dr...,5,6-Feb-11,47
4,26791,Ulipristal,Emergency Contraception,i took ella one 42 hours after having unprotec...,4,17-Jul-17,5


In [6]:
#Load all split data from GCS bucket
train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
val_df = pd.read_csv(f"{DATA_PATH}/val.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")


In [7]:
# Convert to HF Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# CLEAN DATA: Remove null reviews
train_dataset = train_dataset.filter(lambda x: x["review"] is not None and len(str(x["review"]).strip()) > 0)
val_dataset = val_dataset.filter(lambda x: x["review"] is not None and len(str(x["review"]).strip()) > 0)
test_dataset = test_dataset.filter(lambda x: x["review"] is not None and len(str(x["review"]).strip()) > 0)

print(f"Train samples after cleaning: {len(train_dataset)}")
print(f"Val samples after cleaning: {len(val_dataset)}")
print(f"Test samples after cleaning: {len(test_dataset)}")

Filter:   0%|          | 0/129037 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16130 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16130 [00:00<?, ? examples/s]

Train samples after cleaning: 129035
Val samples after cleaning: 16130
Test samples after cleaning: 16130


In [ ]:
print(len(train_df), "→", len(train_dataset))
print(len(val_df), "→", len(val_dataset))
print(len(test_df), "→", len(test_dataset))

129037 → 129035
16130 → 16130
16130 → 16130


In [ ]:
from transformers import pipeline

# 1. Initialize the pipeline
ner_pipeline = pipeline(
    "ner",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple",
    device=0
)

# 2. Refactored function to process groups of reviews at once
def enrich_and_label_batched(examples):
    # This sends all reviews in the batch to the GPU at the same time
    batch_entities = ner_pipeline(examples["review"])
    
    new_reviews = []
    adr_labels = []
    
    for original_text, entities in zip(examples["review"], batch_entities):
        entity_text = " ".join([ent["word"] for ent in entities])
        
        # Check for symptom entities
        has_adr_signal = any(ent["entity_group"] == "Sign_symptom" for ent in entities)
        adr_labels.append(1 if has_adr_signal else 0)
        
        modified_text = original_text + " [ENT] " + entity_text
        new_reviews.append(modified_text)
        
    examples["review"] = new_reviews
    examples["adr_label"] = adr_labels
    return examples

# 3. Apply with batched=True
print("Processing Training Data in fast batches...")
train_dataset = train_dataset.map(enrich_and_label_batched, batched=True, batch_size=8)

print("Processing Validation Data in fast batches...")
val_dataset = val_dataset.map(enrich_and_label_batched, batched=True, batch_size=8)

print("Processing Test Data in fast batches...")
test_dataset = test_dataset.map(enrich_and_label_batched, batched=True, batch_size=8)

print(" All datasets successfully enriched!")

enriched_train_dataset=train_dataset.save_to_disk("gs://adr-nlp/data/processed/train_enriched")
enriched_val_dataset=val_dataset.save_to_disk("gs://adr-nlp/data/processed/val_enriched")
enriched_test_dataset=test_dataset.save_to_disk("gs://adr-nlp/data/processed/test_enriched")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Processing Training Data in fast batches...


Map:   0%|          | 0/129035 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [10]:
from transformers import AutoTokenizer

# 1. Load the tokenizer matching your model
model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define the tokenization function
def tokenize_func(examples):
    return tokenizer(
        examples["review"], 
        truncation=True, 
        padding="max_length", 
        max_length=256
    )

# 3. Apply tokenization to all 3 datasets
print("Tokenizing Train dataset...")
train_dataset = train_dataset.map(tokenize_func, batched=True)

print("Tokenizing Validation dataset...")
val_dataset = val_dataset.map(tokenize_func, batched=True)

print("Tokenizing Test dataset...")
test_dataset = test_dataset.map(tokenize_func, batched=True)

# 4. Target column mapping & formatting
# Hugging Face Trainer expects the label column to be specifically named 'labels'
train_dataset = train_dataset.rename_column("adr_label", "labels")
val_dataset = val_dataset.rename_column("adr_label", "labels")
test_dataset = test_dataset.rename_column("adr_label", "labels")

# Tell PyTorch to strictly look at the numeric tensors it needs
columns_to_keep = ["input_ids", "attention_mask", "labels"]
train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

print("All datasets tokenized and ready for PyTorch!")


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizing Train dataset...


Map:   0%|          | 0/129035 [00:00<?, ? examples/s]

Tokenizing Validation dataset...


Map:   0%|          | 0/16130 [00:00<?, ? examples/s]

Tokenizing Test dataset...


Map:   0%|          | 0/16130 [00:00<?, ? examples/s]

All datasets tokenized and ready for PyTorch!


In [24]:

import numpy as np
import torch
from torch import nn
from transformers import Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

# 1. Dynamically calculate class weights from the training set
print(" Calculating class weights for target balancing...")
labels = np.array(train_dataset["labels"])

weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
weights_tensor = torch.tensor(weights, dtype=torch.float)

# 2. Custom loss function wrapper for class weights
class CustomTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 3. Function to compute classification metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

print("Custom Trainer and Evaluation framework initialized.")


 Calculating class weights for target balancing...
Custom Trainer and Evaluation framework initialized.


In [1]:


from transformers import AutoModelForSequenceClassification, TrainingArguments
import optuna

def model_init(trial):
    return AutoModelForSequenceClassification.from_pretrained(
        "emilyalsentzer/Bio_ClinicalBERT", 
        num_labels=2
    )

def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
    }

args = TrainingArguments(
    output_dir="./hyperparam_search",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)


# We pass both train and validation datasets here!
trainer = CustomTrainer(
    class_weights=weights_tensor,
    model_init=model_init,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset, # Used by Optuna to score and find the best metrics
    compute_metrics=compute_metrics
)

print("Starting Optuna hyperparameter search across 5 runs...")
best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=hp_space,
    n_trials=5
)

print("\nBest hyperparameter run found:")
print(best_run.hyperparameters)


In [ ]:
import mlflow

final_args = TrainingArguments(
    output_dir="./final_biobert_model",
    num_train_epochs=best_run.hyperparameters["num_train_epochs"],
    per_device_train_batch_size=best_run.hyperparameters["per_device_train_batch_size"],
    learning_rate=best_run.hyperparameters["learning_rate"],
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_epoch=True,
    metric_for_best_model="f1"
)

final_trainer = CustomTrainer(
    class_weights=weights_tensor,
    model=model_init(None),
    args=final_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

mlflow.set_experiment("adr-colab-experiment")
with mlflow.start_run() as run:
    mlflow.log_params(best_run.hyperparameters)
    
    print("Training final model...")
    final_trainer.train()
    
    print("\n Running final evaluation on the locked Test Set...")
    test_results = final_trainer.predict(test_dataset)
    
    mlflow.log_metrics(test_results.metrics)
    print("\nFinal Test Metrics:")
    print(test_results.metrics)
    
    print("Saving model locally...")
    final_trainer.save_model("./final_biobert_model")
    tokenizer.save_pretrained("./final_biobert_model")

print("Model trained, evaluated on test set, and saved locally!")


In [ ]:

# SAVE MODEL

trainer.save_model("/content/adr_model")
tokenizer.save_pretrained("/content/adr_model")

mlflow is set up in backend differently for persistent storage

can store in goggle storage
#!mlflow ui --backend-store-uri "gs://adr_nlp/mlruns"  # persistent
#http://localhost:5000 #open to view MLflow UI

Colab as is ephemeral-can view but cannot store!
#import mlflow

#client = mlflow.tracking.MlflowClient()

#experiments = client.search_experiments()
#print(experiments)

